# 知识卡质量与进展分析 · search agent 双 lane（cards / evidence / desc_drafts）

**回答的问题**：双 lane 全量跑批（目标 296,010 实例 × 1 周）现在到哪了、产出的卡质量结构如何、有哪些风险。

**数据源**（全部只读现算，不落盘）：
- `state/taxonomy/search_kb/cards.jsonl` —— 一行一实体一次处理：`status ∈ {ok, reject, llm_error, no_evidence, gate_fail}`，含 `ts/domain/img_count/has_desc/n_evidence/secs/usage`；ok 卡带 `card{definition, desc, canonical_features[{feature, evidence[]}], visual_gaps[]}`，reject 卡带理由
- `state/taxonomy/search_kb/evidence.jsonl` —— 一行一实体一次证据包：`key=实体名`，`rec.evidence[{id, src, title, url, text}]`（**跨批重跑会多行**，478 个 key 有多行）
- `state/taxonomy/search_kb/desc_drafts.jsonl` —— ok 卡的 desc 草稿池（入库方式待拍板的素材，与 ok 行 1:1）
- `state/taxonomy/search_kb/{supervise,run}.{a,b}.log` —— 双 lane 活链日志（只读尾行）
- `datasets/demiwtg/meta/taxonomy.json` —— 域总实例数（首挂载路径 L1 口径，与卡片的 domain 字段同口径）

**口径注意（必读）**：
1. 终态 = 实体最后一行 status ∈ {ok, reject, no_evidence, gate_fail}；末行 llm_error = 未终态（下片重试）。
   进度一律按「唯一实体终态数」算，不看行数。
2. **非法引用校验用并集口径**：实体所有 ok 证据行的 evidence id 并集。重跑实体的卡可能引用早期证据行的 id，
   末行覆盖口径会假阳性（monitor.py 只查最近 10 卡所以没爆过；全量校验必须并集）。
3. 双 lane 域集不相交（lane A 14 域直连 / lane B 15 域走网关，2026-08-30 07:54 起各 c8）；
   域→lane 归属按卡片的 domain 字段（= 首挂载 L1）。

In [ ]:
import json, re, time, random, datetime as dt
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.rcParams['font.family'] = ['Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
random.seed(42); np.random.seed(42)
pd.set_option('display.width', 200)

REPO   = Path('/tank/demiwtg')
SKB    = REPO / 'state' / 'taxonomy' / 'search_kb'
TAX    = REPO / 'datasets' / 'demiwtg' / 'meta' / 'taxonomy.json'
LANE_A = {'人造物体','植物','行为动作','食物','自然景观','属性与状态','声音','材料与物质','时间数量与度量',
          '体育与游戏','数字与互联网文化','政治、法律与社会制度','医学与健康','宗教与信仰'}          # ← lane A 14 域（直连）
LANE_B = {'建筑与基础设施','动物','人物与人体','文化艺术与媒介','场景','文字与信息图形','组织机构与社会事件',
          '交通工具','知识与学科','节日与符号','品牌与产品','真菌与微生物','地理与地点','历史与时代',
          '民族、语言与文化'}                                                                              # ← lane B 15 域（网关）
TERM = {'ok', 'reject', 'no_evidence', 'gate_fail'}

def jsonl(p):
    out = []
    with open(p, encoding='utf-8') as f:
        for l in f:
            l = l.strip()
            if not l: continue
            try: out.append(json.loads(l))
            except json.JSONDecodeError: pass        # 活链追加中的半行，跳过
    return out

rows  = jsonl(SKB / 'cards.jsonl')
drafts= jsonl(SKB / 'desc_drafts.jsonl')
evrows= jsonl(SKB / 'evidence.jsonl')
last  = {}
for r in rows: last[r['entity']] = r                  # 每实体取最后一行
term  = {e: r for e, r in last.items() if r['status'] in TERM}
pool  = defaultdict(set)                              # 并集口径证据池
for v in evrows:
    if v.get('ok'):
        pool[v['key']] |= {e['id'] for e in v['rec'].get('evidence', [])}
tree = json.load(open(TAX))['tree']
dom_total = Counter()
def walk(n):
    segs = n['path'].split(' / ')
    d = segs[1] if len(segs) > 1 else '(根)'
    for inst in n.get('instances') or []: dom_total.setdefault(inst, d)
    for c in n.get('children') or []: walk(c)
walk(tree)
dom_n = Counter(dom_total.values())
key_n = Counter(v['key'] for v in evrows)
print(f"cards {len(rows):,} 行 / 唯一实体 {len(last):,} / 终态 {len(term):,} | 证据 key {len(key_n):,}（多行 {sum(1 for n in key_n.values() if n>1)}）| desc 草稿 {len(drafts):,}")
print(f"域总实例（首挂载 L1）：{sum(dom_n.values()):,}（A {sum(v for k,v in dom_n.items() if k in LANE_A):,} / B {sum(v for k,v in dom_n.items() if k in LANE_B):,}）")

## 1. 进展与速率

- 终态进度按唯一实体；分 lane 用域集归属；速率取近 3h 终态增量（含 slice 重启间隙，比片内「实体/小时」保守）。
- 活链尾行：supervise 片内进度 + run 的 `==` 汇总行（含引擎账本与 down 列表）。

In [ ]:
print('末行状态分布（唯一实体）:', dict(Counter(r['status'] for r in last.values())))
def lane_of(d): return 'A' if d in LANE_A else ('B' if d in LANE_B else '?')
now = time.time()
R3H = 3 * 3600
rec = []
for lane in ('A', 'B', '?'):
    doms = LANE_A if lane == 'A' else (LANE_B if lane == 'B' else set())
    total = sum(v for k, v in dom_n.items() if (k in doms if lane != '?' else k not in LANE_A and k not in LANE_B))
    done  = [r for r in term.values() if lane_of(r['domain']) == lane]
    ok    = [r for r in done if r['status'] == 'ok']
    recent= sum(1 for r in term.values() if lane_of(r['domain']) == lane
                and (now - dt.datetime.fromisoformat(r['ts']).replace(tzinfo=dt.timezone.utc).timestamp()) <= R3H)
    rate  = recent / 3
    rem   = total - len(done)
    rec.append(dict(lane=lane, 目标实例=total, 终态=len(done), ok=len(ok),
                    ok率=f"{len(ok)/max(len(done),1):.0%}", 近3h终态=recent, 速率每小时=round(rate),
                    剩余=rem, ETA天=round(rem / max(rate, 1) / 24, 1) if rate else None))
prog = pd.DataFrame(rec)
display(prog)
print(f"合计：终态 {len(term):,} / 296,010（{len(term)/296010:.1%}） | ok {sum(r['status']=='ok' for r in term.values()):,}"
      f" | 近 3h 合计 {prog['近3h终态'].sum():.0f} 终态/h={prog['速率每小时'].iloc[:2].sum():.0f}"
      f" | 全量 ETA ≈ {(296010-len(term))/max(prog['速率每小时'].iloc[:2].sum(),1)/24:.0f} 天（1 周目标需 ~{(296010-len(term))/7/24:.0f}/h）")

# 每小时终态曲线（近 48h，按 lane 堆叠）
H0 = dt.datetime.now(dt.timezone.utc) - dt.timedelta(hours=48)
hb = {'A': Counter(), 'B': Counter(), '?': Counter()}
for r in term.values():
    t = dt.datetime.fromisoformat(r['ts']).replace(tzinfo=dt.timezone.utc)
    if t >= H0: hb[lane_of(r['domain'])][t.strftime('%m-%d %H')] += 1
hours = sorted({h for c in hb.values() for h in c})
if hours:
    dfh = pd.DataFrame({L: [hb[L].get(h, 0) for h in hours] for L in ('A', 'B', '?')}, index=hours)
    dfh.plot.bar(stacked=True, figsize=(14, 3.2), color=['tab:blue', 'tab:orange', 'gray'],
                 title='每小时终态实体数（近 48h，A/B 按域集堆叠）')
    plt.xticks(rotation=30); plt.tight_layout(); plt.show()
else:
    print('（近 48h 无终态增量——链已停或停超过 48h，曲线略）')


In [ ]:
# 活链尾行（只读）
def tail(path, n=40000):
    if not path.exists(): return ''
    with open(path, 'rb') as f:
        f.seek(0, 2); f.seek(max(0, f.tell() - n)); return f.read().decode('utf-8', 'ignore')
for lane in ('a', 'b'):
    sup = tail(SKB / f'supervise.{lane}.log')
    m = re.findall(r'\[(\d+)/(\d+)\] (\S+)\s+(.+?)\s+\(([\d.]+)s\)', sup)
    if m: print(f'lane {lane.upper()} supervise 片内: {m[-1][0]}/{m[-1][1]} {m[-1][2]} {m[-1][3]}（{m[-1][4]}s/实体）')
    run = tail(SKB / f'run.{lane}.log')
    mm = [l for l in run.splitlines() if '==' in l]
    if mm: print(f'lane {lane.upper()} run 汇总: {mm[-1].split("INFO ")[-1][:240]}')
hs = json.load(open(SKB / 'source_health.json')) if (SKB / 'source_health.json').exists() else {}
print('\n引擎池账本 source_health.json:', json.dumps(hs, ensure_ascii=False)[:600])

## 2. 卡片结构质量（ok 卡全量）

特征/卡、visual_gaps/卡、单锚率（特征只挂 1 条证据）、证据池深、**非法引用全量校验（并集口径，必须 0）**、
img_count / has_desc 联动、GLM token 成本（已花 + 全量推演）。

In [ ]:
oks = [r for r in rows if r['status'] == 'ok' and r['batch'] == 'full'] + \
      [r for r in rows if r['status'] == 'ok' and r['batch'] != 'full']
n_feat = n_bad = n_1ev = 0
feats, gaps, anchors = [], [], []
for r in oks:
    ids = pool.get(r['entity'], set())
    fs = r['card'].get('canonical_features', [])
    feats.append(len(fs)); gaps.append(len(r['card'].get('visual_gaps', [])))
    for f in fs:
        n_feat += 1
        evd = f.get('evidence') or []
        if not set(evd) <= ids: n_bad += 1
        anchors.append(len(evd))
        if len(evd) == 1: n_1ev += 1
print(f"ok 行 {len(oks):,} | 特征/卡 med {np.median(feats):.0f} mean {np.mean(feats):.1f} p10 {np.percentile(feats,10):.0f}"
      f" | gaps/卡 med {np.median(gaps):.0f} | 单锚特征 {n_1ev:,}/{n_feat:,}（{n_1ev/n_feat:.0%}）"
      f" | 非法引用（并集口径） {n_bad} ←必须 0")
pool_deep = [len(v) for v in pool.values()]
print(f"证据池深 med {np.median(pool_deep):.0f} / p90 {np.percentile(pool_deep,90):.0f} | 卡字段 n_evidence 与池深一致率 "
      f"{np.mean([r['n_evidence']==len(pool.get(r['entity'],[])) for r in oks]):.0%}")
imgc = [r.get('img_count') or 0 for r in oks]
print(f"img_count: 0 图实体占 ok 卡 {np.mean([c==0 for c in imgc]):.0%} | has_desc 占比 {np.mean([r.get('has_desc') for r in oks]):.0%}")
tok = sum((r.get('usage') or {}).get('total_tokens', 0) for r in rows)
print(f"GLM tokens 累计 {tok/1e6:.1f}M（均 {tok/len(rows)/1e3:.1f}k/行）→ 全量 296,010 推演 ≈ {tok/len(rows)*296010/1e9:.2f}B tokens")

fig, ax = plt.subplots(1, 3, figsize=(15, 3))
pd.Series(feats).plot.hist(bins=range(0, 16), ax=ax[0], title='特征/卡')
pd.Series(gaps).plot.hist(bins=range(0, 16), ax=ax[1], title='visual_gaps/卡')
pd.Series(anchors).plot.hist(bins=range(1, 6), ax=ax[2], title='每特征证据锚数')
plt.tight_layout(); plt.show()

In [ ]:
# 证据源分布（全量 + 近 200 包）：src 前缀 serp:*=搜索引擎结果页, page=正文抽取, 其余=知识源连接器
src_all, src_recent = Counter(), Counter()
recent_keys = {v['key'] for v in evrows[-200:]}
for v in evrows:
    if not v.get('ok'): continue
    for e in v['rec'].get('evidence', []):
        src_all[e['src']] += 1
        if v['key'] in recent_keys: src_recent[e['src']] += 1
srcd = pd.DataFrame({'全量': src_all, '近200包': src_recent}).fillna(0).astype(int).sort_values('全量', ascending=False)
display(srcd.T)

## 3. 分域进展与质量

- 池先行（`--planner-scope pool`：有合格图的 54,545 实体先跑）造成域倾斜：覆盖最好的域先出卡，不代表全量分布。
- 质量列只对 ok 卡统计；薄卡 = 特征 < 5。

In [ ]:
domstat = defaultdict(lambda: dict(total=0, done=0, ok=0, feats=[], gaps=[], thin=0))
for d, t in dom_n.items(): domstat[d]['total'] = t
for r in term.values():
    d = domstat[r['domain']]; d['done'] += 1
    if r['status'] == 'ok':
        d['ok'] += 1
        fs = r['card'].get('canonical_features', [])
        d['feats'].append(len(fs)); d['gaps'].append(len(r['card'].get('visual_gaps', [])))
        if len(fs) < 5: d['thin'] += 1
ds = pd.DataFrame([dict(domain=d, lane=lane_of(d), total=v['total'], done=v['done'], ok=v['ok'],
                        完成率=round(100*v['done']/max(v['total'],1),1),
                        ok率=round(100*v['ok']/max(v['done'],1),0),
                        特征均=round(np.mean(v['feats']),1) if v['feats'] else None,
                        gaps均=round(np.mean(v['gaps']),1) if v['gaps'] else None,
                        薄卡率=round(100*v['thin']/max(v['ok'],1),1) if v['ok'] else None)
                   for d, v in domstat.items()]).sort_values('done', ascending=False)
display(ds)
unstarted = ds[ds.done == 0]
print(f"未开工域 {len(unstarted)} 个：{list(unstarted['domain'])}")
ds[ds.done > 0].set_index('domain')[['完成率']].plot.barh(figsize=(8, 6), legend=False,
    title='各域完成率（已开工域；done=终态/域总实例）')
plt.tight_layout(); plt.show()

## 4. reject / llm_error / 复活

- reject = 两跳终态（检索证据不足以建立视觉定义，拒绝出卡）——按域看哪些域天然难出卡。
- llm_error 复活率 = 出过 llm_error 但最终 ok/reject 的实体占比（重试机制有效性）；
  末行 llm_error 的实体是未终态（下片自动重试），不算进度也不算失败。

In [ ]:
rej = [r for r in rows if r['status'] == 'reject']
rej_last = [r for r in term.values() if r['status'] == 'reject']
rl = [len(r['card'].get('reject', '')) for r in rej]
print(f"reject 行 {len(rej):,}（终态 {len(rej_last):,}）| 理由长度 med {np.median(rl):.0f} 字")
per = defaultdict(list)
for r in rows: per[r['entity']].append(r['status'])
revived = sum(1 for ss in per.values() if 'llm_error' in ss and ss[-1] in ('ok', 'reject'))
ever_err = sum(1 for ss in per.values() if 'llm_error' in ss)
inflight = [e for e, r in last.items() if r['status'] == 'llm_error']
print(f"llm_error：出过错实体 {ever_err:,}，复活 {revived:,}（{revived/max(ever_err,1):.0%}）| 当前未终态 {len(inflight):,}（下片重试）")
print('终态 reject 按域 top8:', Counter(r['domain'] for r in rej_last).most_common(8))
print('reject 理由抽样：')
for r in random.sample(rej_last, min(3, len(rej_last))):
    print(f"  · [{r['domain']}] {r['entity']}：{r['card'].get('reject', '')[:160]}…")

## 5. 入库素材池（desc_drafts）

ok 卡 1:1 产出 desc 草稿（入库方式待拍板：desc 刷库 / meta/kcards.jsonl / 双写）。
长度对齐 instances.json 的 desc 口径（150-350 字）。

In [ ]:
dl = [len(d['desc']) for d in drafts]
df_ = [len(d.get('definition', '')) for d in drafts]
print(f"desc 草稿 {len(drafts):,} | desc 长度 med {np.median(dl):.0f} | 150-350 字内 {np.mean([150<=x<=350 for x in dl]):.0%}"
      f" | definition 长度 med {np.median(df_):.0f}")
plt.figure(figsize=(7, 2.8))
plt.hist(np.clip(dl, 0, 600), bins=40); plt.axvline(150, color='gray', ls=':'); plt.axvline(350, color='gray', ls=':')
plt.title('desc 草稿长度（虚线=150/350 口径）'); plt.tight_layout(); plt.show()

## 6. 抽样人审（改 SEED 换样本）

抽 ok 卡全文打印（definition + 特征×证据锚 + gaps）+ reject 理由全文，人工核对视觉密度与证据挂靠。

In [ ]:
SEED = 42          # ← 改这里换样本
random.seed(SEED)
S_N, R_N = 3, 2     # ok 卡 / reject 各抽几张
for r in random.sample([x for x in term.values() if x['status'] == 'ok'], S_N):
    c = r['card']
    print(f"\n{'='*70}\n[{r['domain']}] {r['entity']}（img={r.get('img_count')}，证据池 {len(pool.get(r['entity'],[]))} 条，{r['secs']:.0f}s）")
    print('定义:', c.get('definition', ''))
    for f in c.get('canonical_features', []):
        print(f"  · {'+'.join(f.get('evidence') or ['-'])} {f['feature']}")
    print('  gaps:', '；'.join(c.get('visual_gaps', []))[:300])
for r in random.sample([x for x in term.values() if x['status'] == 'reject'], min(R_N, len(rej_last))):
    print(f"\n{'='*70}\n[reject·{r['domain']}] {r['entity']}：{r['card'].get('reject', '')}")

## 6.5 案例增强呈现：配图对照 + 新旧 desc 对比

- 抽样同 §6（SEED2 单独控）：一半抽**已有旧 desc** 的实体（gen_instance_kb·qwen3.8-27B 时代写入 instances.json）做新旧对比，一半抽纯新增实体。
- 每实体取 ≤2 张代表图（合格门 quality≥8 且 identity=true；不足放宽 identity=true，再不足取任意），与新 desc 草稿 + 特征锚并排呈现——人审「检索接地的 desc 是否贴合真实图像」。
- 图走 base64 内嵌（Jupyter 拦 file://），样本量小不影响文件体积。

In [ ]:
import base64, html as _h
import duckdb
from IPython.display import HTML

SEED2 = 42          # ← 改这里换样本
random.seed(SEED2)
N_OLD, N_NEW = 3, 3  # 有旧 desc 对比 / 纯新增 各几张

insts = json.load(open(REPO / 'datasets/demiwtg/meta/instances.json', encoding='utf-8'))['instances']
old = {i['name']: (i.get('desc') or '', i.get('source') or '?') for i in insts}
draft = {d['entity']: d for d in drafts}
cand_old = [r for r in term.values() if r['status'] == 'ok' and r['entity'] in draft and old.get(r['entity'], ('',))[0]]
cand_new = [r for r in term.values() if r['status'] == 'ok' and r['entity'] in draft and not old.get(r['entity'], ('',))[0]]
picks = random.sample(cand_old, min(N_OLD, len(cand_old))) + random.sample(cand_new, min(N_NEW, len(cand_new)))
names = [r['entity'] for r in picks]
print(f'候选：有旧 desc {len(cand_old):,} / 纯新增 {len(cand_new):,}；本轮抽 {len(picks)}')

nl = ','.join(repr(n) for n in names)
imgs = duckdb.sql(f"""
    SELECT inst, sha256, ext, quality, identity FROM (
      SELECT UNNEST(instances) AS inst, sha256, ext, quality, identity
      FROM read_json_auto('{REPO}/datasets/demiwtg/meta/metadata.jsonl'))
    WHERE inst IN ({nl})
""").df()

MIME = {'jpg': 'jpeg', 'jpeg': 'jpeg', 'png': 'png', 'webp': 'webp', 'gif': 'gif'}
def pick_imgs(entity, k=2):
    sub = imgs[imgs.inst == entity].copy()
    if sub.empty: return []
    sub['q'] = sub.quality.fillna(-1)
    ok = sub[(sub.q >= 8) & (sub.identity == True)]
    if len(ok) < k: ok = pd.concat([ok, sub[sub.identity == True]])
    if ok.empty: ok = sub
    return ok.sort_values('q', ascending=False).head(k).itertuples(index=False)

def b64tag(sha, ext, w=300):
    p = REPO / 'datasets/demiwtg/blobs' / sha[:2] / f'{sha}.{ext}'
    if not p.exists(): return ''
    m = MIME.get((ext or '').lower(), 'jpeg')
    return f"<img src=\"data:image/{m};base64,{base64.b64encode(p.read_bytes()).decode()}\" width={w}/>"

frag = []
for r in picks:
    e = r['entity']; od, osrc = old.get(e, ('', '?')); nd = draft[e]
    tags = ''.join(b64tag(t.sha256, t.ext) for t in pick_imgs(e)) or '<i>（湖内无图）</i>'
    feats = ''.join(f"<li><small>[{'+'.join(f.get('evidence') or ['-'])}]</small> {_h.escape(f['feature'])}</li>"
                    for f in r['card'].get('canonical_features', [])[:4])
    oldblk = (f"<hr><b>旧 desc</b>（instances.json · source={osrc} · {len(od)}字）：<blockquote>{_h.escape(od)}</blockquote>"
              if od else '<hr><i>纯新增实体（湖内原无 desc）</i>')
    frag.append(f"""<h4>[{_h.escape(r['domain'])}] {_h.escape(e)} <small>（img={r.get('img_count')}，证据池 {len(pool.get(e, []))} 条，{r['secs']:.0f}s）</small></h4>
<table><tr><td style='vertical-align:top'>{tags}</td><td style='vertical-align:top;max-width:640px'>
<b>新 desc 草稿</b>（{len(nd['desc'])}字）：{_h.escape(nd['desc'])}<br>
<details><summary>特征锚（前4）</summary><ul>{feats}</ul></details>
{oldblk}</td></tr></table>""")
display(HTML('<hr>'.join(frag)))

## 7. 读法速查

- **进展**：第 1 节（终态/分 lane ETA/每小时曲线/活链尾行）——验收判据 = 终态 296,010。
- **质量**：第 2 节结构指标（特征/gaps/单锚/非法引用=0）+ 第 3 节分域 + 第 6 节人眼审。
- **产能风险**：第 1 节速率 vs 1 周目标；第 4 节 llm_error 复活率与未终态存量。
- **入库**：第 5 节 desc 草稿池（长度口径对齐情况），对应交接待拍板项 1。
- 所有参数在各 cell 头部（域集/时间窗/SEED），直接改了重跑；全册只读，不碰双 lane 进程。- **案例人审（图文对照 + 新旧对比）**：第 6.5 节——改 SEED2 换样本，看 desc 与真实图像的贴合度、对比 27B 时代旧 desc 的信息密度。
